In [0]:
# Databricks notebook source
# =============================================================================
# CUSTOMER RISK PROFILE CUBE — Executive Analytics Mart
# =============================================================================
# this mart builds a single row per customer with their complete risk profile.
# it combines:
#   - identity and demographics (from the base enriched table)
#   - account and card portfolio (how many products does this customer have)
#   - merchant exposure (do they shop at risky merchants)
#   - device risk (do they use emulators, rooted phones, shared devices)
#   - fiu risk (from the velocity network mart we already built)
#   - a composite enterprise risk score from 0 to 100
#
# power bi connects directly to this table for executive dashboards.
# =============================================================================

from pyspark.sql.functions import *
from datetime import datetime

# where everything lives
catalog = "ubuntu_bank_fraud"
schema_name = "gold"

base_tbl = f"{catalog}.{schema_name}.gold_base_enriched_transactions"
fiu_tbl  = f"{catalog}.{schema_name}.gold_fiu_customer_velocity_network"
target   = f"{catalog}.{schema_name}.gold_customer_risk_profile_cube"

batch_id = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"base source:  {base_tbl}")
print(f"fiu source:   {fiu_tbl}")
print(f"target:       {target}")
print(f"batch:        {batch_id}")

# load both source tables
base = spark.table(base_tbl)
fiu  = spark.table(fiu_tbl)

# quick counts
base_rows = base.count()
base_custs = base.select("customer_id").distinct().count()
fiu_rows = fiu.count()

print(f"\nbase table: {base_rows:,} rows, {base_custs:,} unique customers")
print(f"fiu table:  {fiu_rows:,} rows")

# make sure the key columns we need actually exist
required_base = ["customer_id", "cust_full_name", "customer_segment",
                 "cust_province", "cust_tenure_bucket", "cust_kyc_score",
                 "source_account_id", "card_id", "merchant_id",
                 "merchant_category", "mcc_risk_weight", "device_id",
                 "dev_risk_score", "is_emulator", "is_rooted",
                 "device_customer_count", "cross_risk_tier"]

missing_base = [c for c in required_base if c not in base.columns]
if missing_base:
    print(f"\n*** missing columns in base table: {missing_base}")
    raise ValueError("cannot continue — required columns missing")
else:
    print(f"\nall {len(required_base)} required base columns present")

required_fiu = ["customer_id", "fiu_risk_score", "fiu_risk_band",
                "total_txns", "total_amount", "inflow_amount", "outflow_amount",
                "rtc_count", "rtc_amount", "night_pct", "weekend_pct",
                "pass_through_flag", "rtc_abuse_flag", "max_others_on_device",
                "device_count", "velocity_growth", "beneficiaries"]

missing_fiu = [c for c in required_fiu if c not in fiu.columns]
if missing_fiu:
    print(f"\n*** missing columns in fiu table: {missing_fiu}")
    raise ValueError("cannot continue — required columns missing")
else:
    print(f"all {len(required_fiu)} required fiu columns present")

base source:  ubuntu_bank_fraud.gold.gold_base_enriched_transactions
fiu source:   ubuntu_bank_fraud.gold.gold_fiu_customer_velocity_network
target:       ubuntu_bank_fraud.gold.gold_customer_risk_profile_cube
batch:        20260623_153854

base table: 2,200,000 rows, 28,311 unique customers
fiu table:  28,311 rows

all 17 required base columns present
all 17 required fiu columns present


In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# pull customer identity columns from the base table
# each customer appears many times in the base table (once per transaction)
# so we take the first non-null value for each identity column
# ---------------------------------------------------------------------------

customer_identity = base.groupBy("customer_id").agg(
    first("cust_full_name", ignorenulls=True).alias("customer_name"),
    first("customer_segment", ignorenulls=True).alias("segment"),
    first("cust_province", ignorenulls=True).alias("province"),
    first("cust_tenure_bucket", ignorenulls=True).alias("tenure_bucket"),
    first("cust_kyc_score", ignorenulls=True).alias("kyc_risk_score"),
    # the most common cross-risk tier across all this customer's transactions
    first("cross_risk_tier", ignorenulls=True).alias("dominant_cross_risk"),
    # when did we first and last see this customer
    min("transaction_date").alias("first_seen"),
    max("transaction_date").alias("last_seen"),
    # how many distinct days were they active
    countDistinct("transaction_date").alias("active_days")
)

print(f"customer identity rows: {customer_identity.count():,}")
customer_identity.select("customer_id", "customer_name", "segment", "province").show(5, truncate=False)

customer identity rows: 28,311
+------------+-----------------+---------------+-------------+
|customer_id |customer_name    |segment        |province     |
+------------+-----------------+---------------+-------------+
|CUST00018663|SANJAY CHETTY    |Mass_Retail    |Western Cape |
|CUST00021170|MUSA DLAMINI     |Mass_Retail    |KwaZulu-Natal|
|CUST00000080|ZANELE BUTHELEZI |Private_Banking|KwaZulu-Natal|
|CUST00018858|THANDI DLAMINI   |Affluent       |Eastern Cape |
|CUST00004328|NOKUTHULA KHUMALO|Premier        |Western Cape |
+------------+-----------------+---------------+-------------+
only showing top 5 rows


In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# build the customer's product portfolio
# how many accounts and cards do they have
# ---------------------------------------------------------------------------

customer_portfolio = base.groupBy("customer_id").agg(
    # how many different accounts
    countDistinct("source_account_id").alias("account_count"),
    # how many different cards (nulls excluded automatically by countDistinct)
    countDistinct("card_id").alias("card_count"),
    # what types of accounts do they hold
    countDistinct("account_type").alias("account_type_diversity"),
    # do they have a credit card (1 = yes, 0 = no)
    max(when(col("account_type") == "Credit_Card", 1).otherwise(0)).alias("has_credit_card"),
    # what is their total credit limit across all accounts
    sum("acct_credit_limit").alias("total_credit_limit")
)

print(f"portfolio rows: {customer_portfolio.count():,}")
customer_portfolio.select("customer_id", "account_count", "card_count", "has_credit_card").show(5)

portfolio rows: 28,311
+------------+-------------+----------+---------------+
| customer_id|account_count|card_count|has_credit_card|
+------------+-------------+----------+---------------+
|CUST00001689|            2|         2|              0|
|CUST00028390|            3|         2|              0|
|CUST00017589|            2|         1|              0|
|CUST00014723|            1|         1|              0|
|CUST00013020|            1|         2|              0|
+------------+-------------+----------+---------------+
only showing top 5 rows


In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# merchant exposure — where does this customer shop and how risky are those places
# a customer who only shops at grocery stores has low merchant risk
# a customer who frequents gaming merchants and high-risk ecommerce has high exposure
# ---------------------------------------------------------------------------

merchant_exposure = base.groupBy("customer_id").agg(
    # how many different merchants visited
    countDistinct("merchant_id").alias("distinct_merchants"),
    # how many different merchant categories
    countDistinct("merchant_category").alias("merchant_category_diversity"),
    # average mcc risk weight across all their merchant transactions
    round(avg("mcc_risk_weight"), 3).alias("avg_mcc_risk_weight"),
    # max mcc risk — what is the riskiest merchant they visit
    max("mcc_risk_weight").alias("max_mcc_risk_weight"),
    # count of transactions at high-risk merchants (mcc risk > 0.50)
    count(when(col("mcc_risk_weight") > 0.50, lit(1))).alias("high_risk_merchant_txns"),
    # how many transactions involve merchants at all (non-null merchant_id)
    count("merchant_id").alias("total_merchant_txns"),
    # count of distinct high-risk merchants (risk band = HIGH)
    countDistinct(when(col("merch_risk_band") == "HIGH", col("merchant_id"))).alias("high_risk_merchant_count")
)

print(f"merchant exposure rows: {merchant_exposure.count():,}")
merchant_exposure.select("customer_id", "distinct_merchants", "avg_mcc_risk_weight").show(5)

merchant exposure rows: 28,311
+------------+------------------+-------------------+
| customer_id|distinct_merchants|avg_mcc_risk_weight|
+------------+------------------+-------------------+
|CUST00018663|                22|              0.266|
|CUST00021170|                38|              0.275|
|CUST00000080|                55|               0.28|
|CUST00018858|                65|              0.294|
|CUST00004328|                26|              0.274|
+------------+------------------+-------------------+
only showing top 5 rows


In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# device risk profile — what devices does this customer use and how risky are they
# key signals: emulators, rooted devices, shared devices, high risk score devices
# ---------------------------------------------------------------------------

device_risk = base.groupBy("customer_id").agg(
    # how many different devices
    countDistinct("device_id").alias("distinct_devices"),
    # worst device risk score across all their devices
    max("dev_risk_score").alias("max_device_risk_score"),
    # average device risk score
    round(avg("dev_risk_score"), 1).alias("avg_device_risk_score"),
    # do any of their devices show emulator flags
    max(when(col("is_emulator") == True, 1).otherwise(0)).alias("uses_emulator"),
    # do any of their devices show rooted flags
    max(when(col("is_rooted") == True, 1).otherwise(0)).alias("uses_rooted_device"),
    # have they used a device for the first time (possible account takeover)
    max(when(col("is_first_txn_on_device") == True, 1).otherwise(0)).alias("has_first_time_device_use"),
    # worst device sharing — max customers on any one of their devices
    max("device_customer_count").alias("max_device_shared_count"),
    # how many fraud_ring linked devices do they use
    count(when(col("dev_link_type") == "FRAUD_RING", lit(1))).alias("fraud_ring_device_links")
)

print(f"device risk rows: {device_risk.count():,}")
device_risk.select("customer_id", "distinct_devices", "uses_emulator", "uses_rooted_device").show(5)

device risk rows: 28,311
+------------+----------------+-------------+------------------+
| customer_id|distinct_devices|uses_emulator|uses_rooted_device|
+------------+----------------+-------------+------------------+
|CUST00018663|               1|            0|                 0|
|CUST00021170|               1|            0|                 0|
|CUST00000080|               1|            0|                 0|
|CUST00018858|               1|            0|                 0|
|CUST00004328|               2|            0|                 0|
+------------+----------------+-------------+------------------+
only showing top 5 rows


In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# stitch all the customer-level pieces into one dataframe
# start with identity, join portfolio, merchant exposure, device risk, and fiu
# ---------------------------------------------------------------------------

# build the full profile by joining all the pieces on customer_id
profile = customer_identity \
    .join(customer_portfolio, "customer_id", "left") \
    .join(merchant_exposure, "customer_id", "left") \
    .join(device_risk, "customer_id", "left") \
    .join(fiu.select(
        "customer_id",
        "fiu_risk_score",
        "fiu_risk_band",
        "total_txns",
        "total_amount",
        "inflow_amount",
        "outflow_amount",
        "rtc_count",
        "rtc_amount",
        "night_pct",
        "weekend_pct",
        "pass_through_flag",
        "rtc_abuse_flag",
        "max_others_on_device",
        "device_count",
        "velocity_growth",
        "beneficiaries"
    ), "customer_id", "left")

# fill nulls with sensible defaults for customers with no data in a category
profile = profile \
    .fillna(0, ["account_count", "card_count", "distinct_merchants",
                "distinct_devices", "total_txns", "total_amount"]) \
    .fillna(0.0, ["avg_mcc_risk_weight", "avg_device_risk_score"])

total_customers = profile.count()
print(f"profile complete: {total_customers:,} customers")
print(f"columns: {len(profile.columns)}")

profile complete: 28,311 customers
columns: 46


In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# ENTERPRISE CUSTOMER RISK SCORE (RECALIBRATED)
# ---------------------------------------------------------------------------
# the previous version gave every customer points just for existing.
# this version only awards points when behaviour genuinely deviates from normal.
#
# key changes:
#   1. each component has a "normal baseline" — below this, score = 0
#   2. only deviation above the baseline contributes to risk
#   3. pass-through is now a dedicated component (strong mule signal)
#   4. final bands use percentile thresholds, not absolute cutoffs
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# first, figure out the median values for baseline zeroing
# the median represents "normal" — half of customers are below this
# ---------------------------------------------------------------------------

medians = profile.approxQuantile(
    ["fiu_risk_score", "total_txns", "night_pct", "weekend_pct",
     "avg_mcc_risk_weight", "distinct_merchants", "max_device_risk_score"],
    [0.5], 0.01
)

median_fiu      = medians[0][0]
median_txns     = medians[1][0]
median_night    = medians[2][0]
median_weekend  = medians[3][0]
median_mcc      = medians[4][0]
median_merch    = medians[5][0]
median_dev_risk = medians[6][0]

print("population medians for baseline zeroing:")
print(f"  fiu score:          {median_fiu:.1f}")
print(f"  total transactions: {median_txns:.0f}")
print(f"  night pct:          {median_night:.3f}")
print(f"  weekend pct:        {median_weekend:.3f}")
print(f"  avg mcc weight:     {median_mcc:.3f}")
print(f"  distinct merchants: {median_merch:.0f}")
print(f"  max device risk:    {median_dev_risk:.1f}")

# also need maximums for scaling above the baseline
max_vals = profile.agg(
    max("fiu_risk_score").alias("max_fiu"),
    max("avg_mcc_risk_weight").alias("max_mcc"),
    max("distinct_merchants").alias("max_merch"),
    max("max_device_risk_score").alias("max_dev_risk"),
    max("night_pct").alias("max_night"),
    max("weekend_pct").alias("max_weekend"),
    max("velocity_growth").alias("max_growth"),
    max("total_txns").alias("max_txns")
).collect()[0]

# ---------------------------------------------------------------------------
# COMPONENT 1: fiu risk (0 to 25 points)
# baseline = median fiu score. only scores ABOVE the median contribute.
# ---------------------------------------------------------------------------

profile = profile.withColumn(
    "scr_ent_fiu",
    when(
        (col("fiu_risk_score") > median_fiu) & (lit(max_vals["max_fiu"]) > median_fiu),
        round(((col("fiu_risk_score") - median_fiu) /
               (lit(max_vals["max_fiu"]) - median_fiu)) * 25, 1)
    ).otherwise(0)
)

# ---------------------------------------------------------------------------
# COMPONENT 2: merchant exposure (0 to 15 points)
# only contributes if avg mcc weight is above the population median
# or if the customer visits unusually many distinct merchants
# ---------------------------------------------------------------------------

profile = profile \
    .withColumn(
        "scr_ent_mcc",
        when(col("avg_mcc_risk_weight") > median_mcc,
             round(((col("avg_mcc_risk_weight") - median_mcc) /
                    (lit(max_vals["max_mcc"]) - median_mcc)) * 8, 1))
        .otherwise(0)
    ) \
    .withColumn(
        "scr_ent_merch_div",
        when(col("distinct_merchants") > median_merch,
             round(((col("distinct_merchants") - median_merch) /
                    (lit(max_vals["max_merch"]) - median_merch)) * 7, 1))
        .otherwise(0)
    ) \
    .withColumn(
        "scr_ent_merchant",
        col("scr_ent_mcc") + col("scr_ent_merch_div")
    )

# ---------------------------------------------------------------------------
# COMPONENT 3: device risk (0 to 20 points)
# emulator = automatic 10, rooted = automatic 5
# high device risk score (above median) = up to 5 more
# ---------------------------------------------------------------------------

profile = profile \
    .withColumn(
        "scr_ent_emu",
        when(col("uses_emulator") == 1, 10).otherwise(0)
    ) \
    .withColumn(
        "scr_ent_root",
        when(col("uses_rooted_device") == 1, 5).otherwise(0)
    ) \
    .withColumn(
        "scr_ent_dev_score",
        when(col("max_device_risk_score") > median_dev_risk,
             round(((col("max_device_risk_score") - median_dev_risk) /
                    (lit(max_vals["max_dev_risk"]) - median_dev_risk)) * 5, 1))
        .otherwise(0)
    ) \
    .withColumn(
        "scr_ent_device",
        col("scr_ent_emu") + col("scr_ent_root") + col("scr_ent_dev_score")
    )

# ---------------------------------------------------------------------------
# COMPONENT 4: behaviour risk (0 to 15 points)
# only contributes if night or weekend activity is above the median
# ---------------------------------------------------------------------------

profile = profile \
    .withColumn(
        "scr_ent_night",
        when(col("night_pct") > median_night,
             round(((col("night_pct") - median_night) /
                    (lit(max_vals["max_night"]) - median_night)) * 10, 1))
        .otherwise(0)
    ) \
    .withColumn(
        "scr_ent_weekend",
        when(col("weekend_pct") > median_weekend,
             round(((col("weekend_pct") - median_weekend) /
                    (lit(max_vals["max_weekend"]) - median_weekend)) * 5, 1))
        .otherwise(0)
    ) \
    .withColumn(
        "scr_ent_behaviour",
        col("scr_ent_night") + col("scr_ent_weekend")
    )

# ---------------------------------------------------------------------------
# COMPONENT 5: activity risk (0 to 10 points)
# only contributes if transaction volume is above the median
# ---------------------------------------------------------------------------

profile = profile.withColumn(
    "scr_ent_activity",
    when(col("total_txns") > median_txns,
         round(((col("total_txns") - median_txns) /
                (lit(max_vals["max_txns"]) - median_txns)) * 10, 1))
    .otherwise(0)
)

# ---------------------------------------------------------------------------
# COMPONENT 6: pass-through risk (0 to 15 points) — NEW
# a dedicated mule indicator based on the pass_through_flag and ratio
# ---------------------------------------------------------------------------

profile = profile.withColumn(
    "scr_ent_pass_through",
    when(col("pass_through_flag") == 1, 15)
    .when(col("inflow_amount") > 100000, 8)
    .otherwise(0)
)

# ---------------------------------------------------------------------------
# COMBINE INTO RAW SCORE
# ---------------------------------------------------------------------------

profile = profile \
    .withColumn(
        "enterprise_risk_score_raw",
        col("scr_ent_fiu") +
        col("scr_ent_merchant") +
        col("scr_ent_device") +
        col("scr_ent_behaviour") +
        col("scr_ent_activity") +
        col("scr_ent_pass_through")
    ) \
    .withColumn(
        "enterprise_risk_score",
        round(least(col("enterprise_risk_score_raw"), lit(100)), 0).cast("int")
    )

# ---------------------------------------------------------------------------
# PERCENTILE-BASED RISK BANDS
# instead of hardcoded thresholds, we find the score at each percentile
# and assign bands based on where the customer falls in the population
# ---------------------------------------------------------------------------

# calculate the score boundaries
score_boundaries = profile.approxQuantile("enterprise_risk_score",
                                           [0.55, 0.80, 0.95], 0.01)

low_cutoff    = score_boundaries[0]  # 55th percentile
medium_cutoff = score_boundaries[1]  # 80th percentile
high_cutoff   = score_boundaries[2]  # 95th percentile

print(f"\npercentile-based band thresholds:")
print(f"  LOW      → 0 to {low_cutoff:.0f}  (bottom 55%)")
print(f"  MEDIUM   → {low_cutoff:.0f} to {medium_cutoff:.0f}  (55th-80th)")
print(f"  HIGH     → {medium_cutoff:.0f} to {high_cutoff:.0f}  (80th-95th)")
print(f"  CRITICAL → {high_cutoff:.0f}+  (top 5%)")

profile = profile.withColumn(
    "enterprise_risk_band",
    when(col("enterprise_risk_score") >= high_cutoff, "CRITICAL")
    .when(col("enterprise_risk_score") >= medium_cutoff, "HIGH")
    .when(col("enterprise_risk_score") >= low_cutoff, "MEDIUM")
    .otherwise("LOW")
)

# ---------------------------------------------------------------------------
# SHOW THE NEW DISTRIBUTION
# ---------------------------------------------------------------------------

print("\nrecalibrated risk band distribution:")
profile.groupBy("enterprise_risk_band").count().orderBy(col("count").desc()).show()

print("recalibrated score statistics:")
profile.select("enterprise_risk_score").describe().show()

population medians for baseline zeroing:
  fiu score:          28.0
  total transactions: 58
  night pct:          1.000
  weekend pct:        0.283
  avg mcc weight:     0.287
  distinct merchants: 32
  max device risk:    54.0

percentile-based band thresholds:
  LOW      → 0 to 14  (bottom 55%)
  MEDIUM   → 14 to 21  (55th-80th)
  HIGH     → 21 to 30  (80th-95th)
  CRITICAL → 30+  (top 5%)

recalibrated risk band distribution:
+--------------------+-----+
|enterprise_risk_band|count|
+--------------------+-----+
|                 LOW|14875|
|              MEDIUM| 6822|
|                HIGH| 4662|
|            CRITICAL| 1952|
+--------------------+-----+

recalibrated score statistics:
+-------+---------------------+
|summary|enterprise_risk_score|
+-------+---------------------+
|  count|                28311|
|   mean|   15.820882342552364|
| stddev|    7.447417685258638|
|    min|                    0|
|    max|                   51|
+-------+---------------------+



In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# write the customer risk profile cube to gold
# ---------------------------------------------------------------------------

target = f"{catalog}.{schema_name}.gold_customer_risk_profile_cube"

final = profile.select(

    # ---- identity ----
    "customer_id",
    "customer_name",
    "segment",
    "province",
    "tenure_bucket",
    "kyc_risk_score",
    "dominant_cross_risk",
    "first_seen",
    "last_seen",
    "active_days",

    # ---- product portfolio ----
    "account_count",
    "card_count",
    "account_type_diversity",
    "has_credit_card",
    "total_credit_limit",

    # ---- merchant exposure ----
    "distinct_merchants",
    "merchant_category_diversity",
    "avg_mcc_risk_weight",
    "max_mcc_risk_weight",
    "high_risk_merchant_txns",
    "total_merchant_txns",
    "high_risk_merchant_count",

    # ---- device risk ----
    "distinct_devices",
    "max_device_risk_score",
    "avg_device_risk_score",
    "uses_emulator",
    "uses_rooted_device",
    "has_first_time_device_use",
    "max_device_shared_count",
    "fraud_ring_device_links",

    # ---- fiu risk (from velocity network) ----
    "fiu_risk_score",
    "fiu_risk_band",
    "total_txns",
    "total_amount",
    "inflow_amount",
    "outflow_amount",
    "rtc_count",
    "rtc_amount",
    "night_pct",
    "weekend_pct",
    "pass_through_flag",
    "rtc_abuse_flag",
    "max_others_on_device",
    "device_count",
    "velocity_growth",
    "beneficiaries",

    # ---- enterprise score components ----
    "scr_ent_fiu",
    "scr_ent_merchant",
    "scr_ent_device",
    "scr_ent_behaviour",
    "scr_ent_activity",
    "scr_ent_pass_through",

    # ---- final risk ----
    "enterprise_risk_score",
    "enterprise_risk_band",

    # ---- audit ----
    lit(batch_id).alias("batch_id"),
    current_timestamp().alias("built_at")
)

print(f"writing to {target}")
print(f"rows: {final.count():,}")
print(f"columns: {len(final.columns)}")

final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target)

print("done — table written")

writing to ubuntu_bank_fraud.gold.gold_customer_risk_profile_cube
rows: 28,311
columns: 56
done — table written


In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# validate the customer risk profile cube
# ---------------------------------------------------------------------------

target = f"{catalog}.{schema_name}.gold_customer_risk_profile_cube"
tbl = spark.table(target)

print("=" * 50)
print("VALIDATION: gold_customer_risk_profile_cube")
print("=" * 50)

# 1. row count
total = tbl.count()
print(f"\nrows: {total:,}")

# 2. no duplicates
dupes = tbl.groupBy("customer_id").count().filter(col("count") > 1).count()
print(f"duplicate customers: {dupes} {'OK' if dupes == 0 else 'FAIL'}")

# 3. null customer ids
nulls = tbl.filter(col("customer_id").isNull()).count()
print(f"null customer ids: {nulls} {'OK' if nulls == 0 else 'FAIL'}")

# 4. risk band distribution — this is the key check
print("\nenterprise risk band distribution:")
band_dist = tbl.groupBy("enterprise_risk_band").count().orderBy(col("count").desc())
band_dist.show()

# 5. realism checks
low_pct = tbl.filter(col("enterprise_risk_band") == "LOW").count() / total * 100
crit_pct = tbl.filter(col("enterprise_risk_band") == "CRITICAL").count() / total * 100
print(f"LOW: {low_pct:.1f}%  {'OK' if low_pct > 40 else 'CHECK — expected >40%'}")
print(f"CRITICAL: {crit_pct:.1f}%  {'OK' if crit_pct < 8 else 'CHECK — expected <8%'}")

# 6. score stats
print("\nscore statistics:")
tbl.select("enterprise_risk_score").describe().show()

# 7. segment breakdown
print("\nrisk by customer segment:")
tbl.groupBy("segment").agg(
    round(avg("enterprise_risk_score"), 1).alias("avg_score"),
    count("*").alias("customers")
).orderBy(col("avg_score").desc()).show()

# 8. top 10 critical
print("\ntop 10 critical risk customers:")
tbl.filter(col("enterprise_risk_band") == "CRITICAL") \
    .select("customer_id", "customer_name", "segment", "enterprise_risk_score",
            "fiu_risk_score", "uses_emulator", "pass_through_flag") \
    .orderBy(col("enterprise_risk_score").desc()) \
    .show(10, truncate=False)

print("\n" + "=" * 50)
print("validation complete")
print("=" * 50)

VALIDATION: gold_customer_risk_profile_cube

rows: 28,311
duplicate customers: 0 OK
null customer ids: 0 OK

enterprise risk band distribution:
+--------------------+-----+
|enterprise_risk_band|count|
+--------------------+-----+
|                 LOW|14875|
|              MEDIUM| 6822|
|                HIGH| 4662|
|            CRITICAL| 1952|
+--------------------+-----+

LOW: 52.5%  OK
CRITICAL: 6.9%  OK

score statistics:
+-------+---------------------+
|summary|enterprise_risk_score|
+-------+---------------------+
|  count|                28311|
|   mean|   15.821341528027975|
| stddev|    7.447108555480882|
|    min|                    0|
|    max|                   51|
+-------+---------------------+


risk by customer segment:
+---------------+---------+---------+
|        segment|avg_score|customers|
+---------------+---------+---------+
|Private_Banking|     23.6|     1496|
|        Premier|     19.9|     2802|
|       Affluent|     17.0|     5665|
|    Mass_Retail|     14.2